<a href="https://colab.research.google.com/github/Ugradcoder-ys/mangan-ji-pepper-counting/blob/main/notebooks/rtdetr_boosttrack_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#環境構築

In [ ]:
# 必要なライブラリのインストール
# ultralytics: YOLOv11用
# deep-sort-realtime: DeepSORTの実装ライブラリ
# roboflow: データセットダウンロード用 (追加)
!pip install ultralytics deep-sort-realtime roboflow

# Google Driveのマウント（データセットや動画の保存先）
from google.colab import drive
drive.mount('/content/drive')

#動画のフォーマット修正

In [ ]:
import os

# ===== 設定 =====
SRC_DIR = "/content/drive/MyDrive/testmovies"
OUT_DIR = "/content/drive/MyDrive/testmovies_fixed"
FPS = 30

os.makedirs(OUT_DIR, exist_ok=True)

# ===== 動画一括変換 =====
for fname in sorted(os.listdir(SRC_DIR)):
    if not fname.lower().endswith(".mp4"):
        continue

    src_path = os.path.join(SRC_DIR, fname)

    # 出力ファイル名（_fixed を付与）
    base, ext = os.path.splitext(fname)
    out_path = os.path.join(OUT_DIR, f"{base}_fixed{ext}")

    print(f"変換開始: {fname}")

    cmd = f'''
    ffmpeg -y -i "{src_path}" \
    -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2,fps={FPS}" \
    -c:v libx264 -pix_fmt yuv420p -r {FPS} \
    -movflags +faststart -profile:v high -g {FPS} -bf 2 \
    "{out_path}"
    '''
    os.system(cmd)

    print(f"変換完了: {out_path}")

print("すべての動画変換が完了しました")


# RT-DETR + BoostTrack 実装（追跡・カウント）

In [ ]:
!mkdir -p /content/drive/MyDrive/reid

In [ ]:
!wget -O /content/drive/MyDrive/reid/osnet_x0_25_msmt17.pt \
https://github.com/mikel-brostrom/yolo_tracking/releases/download/v0.1.0/osnet_x0_25_msmt17.pt

In [ ]:
from pathlib import Path
print(Path("/content/drive/MyDrive/reid/osnet_x0_25_msmt17.pt").exists())

In [ ]:
# ============================================================
# RT-DETR (Ultralytics) + BoostTrack (BoxMOT) + Counting
# - Tracking: BoostTrack
# - Output: counted video + final counts print
# ============================================================

# 必要ならインストール（未実施の場合のみ）
# !pip install ultralytics boxmot opencv-python

import cv2
import numpy as np
import torch
from pathlib import Path

from ultralytics import RTDETR
from boxmot import BoostTrack

# ==========================================
# 1. 設定
# ==========================================
MODEL_PATH  = "/content/drive/MyDrive/manganji_rtdetr_best.pt"
VIDEO_PATH  = "/content/drive/MyDrive/testmovies_fixed/testmovie1.1_fixed.MP4"
OUTPUT_PATH = "/content/drive/MyDrive/testmovies_boosttrack_counted.mp4"

# クラス定義 (0: Dried, 1: Flower, 2: Fruit と仮定)
CLASS_NAMES = {0: "dried_flowers", 1: "flowers", 2: "fruits"}

# 色（BGR）
COLORS = {0: (92, 113, 166), 1: (220, 220, 255), 2: (0, 255, 0)}

CONF_THRESHOLD = 0.45

# ReIDモデル重み（必ず存在するパスにしてください）
REID_WEIGHTS = "/content/drive/MyDrive/reid/osnet_x0_25_msmt17.pt"

# ==========================================
# 2. 初期化
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

print(f"モデルロード中: {MODEL_PATH}")
model = RTDETR(MODEL_PATH)

reid_weights_path = Path(REID_WEIGHTS)
assert reid_weights_path.exists(), f"ReID重みが見つかりません: {reid_weights_path}"

tracker = BoostTrack(
    reid_weights=reid_weights_path,
    device=device,
    half=False,  # fp16にしたいなら True（挙動確認後に）
)

# カウント：クラスごとのユニークID集合
unique_id_sets = {cid: set() for cid in CLASS_NAMES.keys()}

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"動画を開けません: {VIDEO_PATH}"

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

writer = cv2.VideoWriter(
    OUTPUT_PATH,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h),
)

print(f"処理開始: {w}x{h} @ {fps}fps")

# ==========================================
# 3. メインループ
# ==========================================
frame_cnt = 0

with torch.inference_mode():
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # --- RT-DETR 推論 ---
        results = model(frame, verbose=False)[0]

        # --- BoostTrackに渡す dets_array を作成 ---
        # 形式: (N,6) [x1,y1,x2,y2,conf,cls_id]
        dets_list = []
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])

            if conf >= CONF_THRESHOLD and cls_id in CLASS_NAMES:
                dets_list.append([x1, y1, x2, y2, conf, cls_id])

        dets_array = (
            np.array(dets_list, dtype=np.float32)
            if len(dets_list) > 0
            else np.empty((0, 6), dtype=np.float32)
        )

        # --- BoostTrack 更新 ---
        # 返り値（想定）: (M,8) [x1,y1,x2,y2,track_id,conf,cls_id,ind]
        tracks = tracker.update(dets_array, frame)
        tracks = np.asarray(tracks) if tracks is not None else np.empty((0, 8), dtype=np.float32)

        # --- 追跡・カウント・描画 ---
        if tracks.shape[0] > 0:
            for tr in tracks:
                x1, y1, x2, y2 = map(int, tr[0:4])
                track_id = int(tr[4])
                cls_id = int(tr[6])

                # カウント（クラス別ユニークID）
                if cls_id in unique_id_sets:
                    unique_id_sets[cls_id].add(track_id)

                color = COLORS.get(cls_id, (255, 255, 255))
                cls_name = CLASS_NAMES.get(cls_id, "unknown")

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(
                    frame,
                    f"{cls_name} id:{track_id}",
                    (x1, max(0, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2,
                )

        # --- 画面へのカウント表示 ---
        overlay = frame.copy()
        cv2.rectangle(overlay, (5, 5), (320, 115), (0, 0, 0), -1)
        frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

        y = 30
        cv2.putText(
            frame,
            "Counts (unique IDs):",
            (10, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
        )

        for cid in sorted(CLASS_NAMES.keys()):
            y += 25
            txt = f"{CLASS_NAMES[cid]}: {len(unique_id_sets[cid])}"
            cv2.putText(
                frame,
                txt,
                (20, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                COLORS.get(cid, (255, 255, 255)),
                2,
            )

        writer.write(frame)

        frame_cnt += 1
        if frame_cnt % 50 == 0:
            print(
                f"processed: {frame_cnt} frames | "
                + ", ".join([f"{CLASS_NAMES[c]}={len(unique_id_sets[c])}" for c in sorted(CLASS_NAMES.keys())])
            )

cap.release()
writer.release()
print(f"完了！動画を保存しました: {OUTPUT_PATH}")

# ==========================================
# 4. 最終個数の出力（指定フォーマット）
# ==========================================
print("\n===== Final Counts =====")
for cid in sorted(CLASS_NAMES.keys()):
    print(f"{CLASS_NAMES[cid]}:{len(unique_id_sets[cid])}")
print("========================\n")


##一括処理ver


In [ ]:
# ============================================================
# Batch: RT-DETR (Ultralytics) + BoostTrack (BoxMOT) + Counting
# Input : /content/drive/MyDrive/testmovies_fixed/*.mp4(MP4)
# Output: /content/drive/MyDrive/testmovies_boosttrack_counted/{name}_counted.mp4
# Print : final counts per video
# ============================================================

# 必要ならインストール（未実施の場合のみ）
!pip install ultralytics boxmot opencv-python

import os
import cv2
import numpy as np
import torch
from pathlib import Path

from ultralytics import RTDETR
from boxmot import BoostTrack

# =========================
# 1) 設定
# =========================
MODEL_PATH = "/content/drive/MyDrive/manganji_rtdetr_best.pt"

SRC_DIR = "/content/drive/MyDrive/testmovies_fixed"
OUT_DIR = "/content/drive/MyDrive/testmovies_boosttrack_counted"
os.makedirs(OUT_DIR, exist_ok=True)

# クラス定義 (0: Dried, 1: Flower, 2: Fruit と仮定)
CLASS_NAMES = {0: "dried_flowers", 1: "flowers", 2: "fruits"}

# 色（BGR）
COLORS = {0: (92, 113, 166), 1: (220, 220, 255), 2: (0, 255, 0)}

#CONF_THRESHOLD = 0.45
#CONF_THRESHOLD = 0.60
CONF_THRESHOLD = 0.75
#CONF_THRESHOLD = 0.90

# ReIDモデル重み（必ず存在するパスにしてください）
REID_WEIGHTS = "/content/drive/MyDrive/reid/osnet_x0_25_msmt17.pt"

# 対象拡張子（mp4 / MP4）
VIDEO_EXTS = (".mp4", ".MP4")

# =========================
# 2) 初期化（モデルは一度だけ）
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

print(f"モデルロード中: {MODEL_PATH}")
model = RTDETR(MODEL_PATH)

reid_weights_path = Path(REID_WEIGHTS)
assert reid_weights_path.exists(), f"ReID重みが見つかりません: {reid_weights_path}"

tracker = BoostTrack(
    reid_weights=reid_weights_path,
    device=device,
    half=False,  # fp16にしたいなら True（挙動確認後に）
)

# =========================
# 3) 入力動画リスト取得
# =========================
src_dir = Path(SRC_DIR)
assert src_dir.exists(), f"入力フォルダが存在しません: {SRC_DIR}"

video_paths = sorted([p for p in src_dir.iterdir() if p.suffix in VIDEO_EXTS or p.name.lower().endswith(".mp4")])
assert len(video_paths) > 0, f"動画が見つかりません: {SRC_DIR} (mp4/MP4)"

print(f"対象動画数: {len(video_paths)}")
for p in video_paths:
    print(" -", p.name)

# =========================
# 4) 動画ごとに処理
# =========================
for idx, video_path in enumerate(video_paths, start=1):
    # 動画ごとの出力
    out_path = Path(OUT_DIR) / f"{video_path.stem}_counted.mp4"

    # カウント：クラスごとのユニークID集合（動画ごとにリセット）
    unique_id_sets = {cid: set() for cid in CLASS_NAMES.keys()}

    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"動画を開けません: {video_path}"

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    writer = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h),
    )

    print("\n" + "="*60)
    print(f"[{idx}/{len(video_paths)}] 処理開始: {video_path.name} | {w}x{h} @ {fps:.2f}fps")
    print("="*60)

    frame_cnt = 0

    # ※重要：trackerの内部状態を動画ごとにクリアしたい場合がある
    # BoostTrack が reset を持っている環境なら呼ぶ（無ければそのまま続行）
    if hasattr(tracker, "reset") and callable(getattr(tracker, "reset")):
        tracker.reset()

    with torch.inference_mode():
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # --- RT-DETR 推論 ---
            results = model(frame, verbose=False)[0]

            # --- BoostTrackに渡す dets_array を作成 ---
            # 形式: (N,6) [x1,y1,x2,y2,conf,cls_id]
            dets_list = []
            for box in results.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])

                if conf >= CONF_THRESHOLD and cls_id in CLASS_NAMES:
                    dets_list.append([x1, y1, x2, y2, conf, cls_id])

            dets_array = (
                np.array(dets_list, dtype=np.float32)
                if len(dets_list) > 0
                else np.empty((0, 6), dtype=np.float32)
            )

            # --- BoostTrack 更新 ---
            # 返り値（想定）: (M,8) [x1,y1,x2,y2,track_id,conf,cls_id,ind]
            tracks = tracker.update(dets_array, frame)
            tracks = np.asarray(tracks) if tracks is not None else np.empty((0, 8), dtype=np.float32)

            # --- 追跡・カウント・描画 ---
            if tracks.shape[0] > 0:
                for tr in tracks:
                    x1, y1, x2, y2 = map(int, tr[0:4])
                    track_id = int(tr[4])
                    cls_id = int(tr[6])

                    # カウント（クラス別ユニークID）
                    if cls_id in unique_id_sets:
                        unique_id_sets[cls_id].add(track_id)

                    color = COLORS.get(cls_id, (255, 255, 255))
                    cls_name = CLASS_NAMES.get(cls_id, "unknown")

                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(
                        frame,
                        f"{cls_name} id:{track_id}",
                        (x1, max(0, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        color,
                        2,
                    )

            # --- 画面へのカウント表示 ---
            overlay = frame.copy()
            cv2.rectangle(overlay, (5, 5), (320, 115), (0, 0, 0), -1)
            frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

            y = 30
            cv2.putText(
                frame,
                "Counts (unique IDs):",
                (10, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2,
            )

            for cid in sorted(CLASS_NAMES.keys()):
                y += 25
                txt = f"{CLASS_NAMES[cid]}: {len(unique_id_sets[cid])}"
                cv2.putText(
                    frame,
                    txt,
                    (20, y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    COLORS.get(cid, (255, 255, 255)),
                    2,
                )

            writer.write(frame)

            frame_cnt += 1
            if frame_cnt % 50 == 0:
                print(
                    f"processed: {frame_cnt} frames | "
                    + ", ".join([f"{CLASS_NAMES[c]}={len(unique_id_sets[c])}" for c in sorted(CLASS_NAMES.keys())])
                )

    cap.release()
    writer.release()

    print(f"完了！動画を保存しました: {out_path}")

    # --- 最終個数の出力（指定フォーマット） ---
    print("\n===== Final Counts =====")
    print(f"video:{video_path.name}")
    for cid in sorted(CLASS_NAMES.keys()):
        print(f"{CLASS_NAMES[cid]}:{len(unique_id_sets[cid])}")
    print("========================\n")

print("全動画の処理が完了しました。")
